In [1]:
import torch
import torchvision.transforms as transforms
import torchvision.transforms.functional as F
import torch.nn as nn
import torch.nn.functional as F_nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import fetch_lfw_people
from torch.utils.data import Dataset, DataLoader
import random
import sqlite3
import pickle
from PIL import Image
from sklearn.model_selection import train_test_split

In [2]:
print("Downloading dataset via scikit-learn...")
lfw = fetch_lfw_people(min_faces_per_person = 5, color = True, resize=1.0, slice_=None)
images = lfw.images
labels = lfw.target
names = lfw.target_names
print(f"Successfully loaded {len(images)} images!")
print(f"Raw Numpy Shape: {images.shape} (N, H, W, C)")

Successfully loaded 5985 images!
Raw Numpy Shape: (5985, 250, 250, 3) (N, H, W, C)


In [3]:
images_transposed = np.transpose(images, (0, 3, 1, 2))

tensor_images = torch.tensor(images_transposed).float()

tensor_images = F.resize(tensor_images, size=[128, 128])

X_train, X_test, y_train, y_test = train_test_split(
    tensor_images, labels, test_size=0.20, random_state=42, stratify=labels
)

print(f"Training Images: {len(X_train)}")
print(f"Testing Images: {len(X_test)}")

Training Images: 4788
Testing Images: 1197


In [4]:
class TripleFaceDataset(Dataset):
    def __init__(self, tensor_images, labels):
        self.images = tensor_images
        self.labels = labels
        self.unique_labels = np.unique(labels)
        # Label mapped with respective indices
        self.label_to_indices = {label: np.where(self.labels == label)[0] for label in self.unique_labels}
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, index):
        anchor_img = self.images[index]
        anchor_label = self.labels[index]

        positive_idx = random.choice(self.label_to_indices[anchor_label])
        positive_img = self.images[positive_idx]

        negative_label = random.choice(labels)
        while(negative_label == anchor_label):
            negative_label = random.choice(labels)
        
        negative_idx = random.choice(self.label_to_indices[negative_label])
        negative_img = self.images[negative_idx]

        return anchor_img, positive_img, negative_img



In [5]:
train_dataset = TripleFaceDataset(X_train, y_train)
test_dataset = TripleFaceDataset(X_test, y_test)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, drop_last=False)

print(f"Train Batches: {len(train_dataloader)}")
print(f"Test Batches: {len(test_dataloader)}")


Train Batches: 149
Test Batches: 38


In [6]:
class FaceEmbeddingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels = 64, out_channels = 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc = nn.Linear(in_features = 128*16*16, out_features = 128)
        
    def forward(self, x):

        x = self.pool(F_nn.relu(self.conv1(x)))
        x = self.pool(F_nn.relu(self.conv2(x)))
        x = self.pool(F_nn.relu(self.conv3(x)))

        x = torch.flatten(x, start_dim=1)

        x = self.fc(x)

        x = F_nn.normalize(x, p=2, dim=1)

        return x

print("FaceNet Architecture Ready")

        

FaceNet Architecture Ready


In [7]:
model = FaceEmbeddingNet()

criterion = nn.TripletMarginLoss(margin = 1.0, p = 2)

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [8]:
epochs = 30

for epoch in range(epochs):
    total_loss = 0.0

    for batch_idx, (anchor, positive, negative) in enumerate(train_dataloader):
        optimizer.zero_grad()

        emb_anchor = model(anchor)
        emb_positive = model(positive)
        emb_negative = model(negative)

        loss = criterion(emb_anchor, emb_positive, emb_negative)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss/len(train_dataloader)
    print(f"Epoch {epoch+1}/{epochs} - Average Loss: {avg_loss:.4f}")

print("Training Finished! The model can now recognize faces!")

Epoch 1/30 - Average Loss: 0.7150
Epoch 2/30 - Average Loss: 0.5744
Epoch 3/30 - Average Loss: 0.5221
Epoch 4/30 - Average Loss: 0.4811
Epoch 5/30 - Average Loss: 0.4484
Epoch 6/30 - Average Loss: 0.4121
Epoch 7/30 - Average Loss: 0.3976
Epoch 8/30 - Average Loss: 0.3617
Epoch 9/30 - Average Loss: 0.3371
Epoch 10/30 - Average Loss: 0.3271
Epoch 11/30 - Average Loss: 0.3092
Epoch 12/30 - Average Loss: 0.2936
Epoch 13/30 - Average Loss: 0.2933
Epoch 14/30 - Average Loss: 0.2758
Epoch 15/30 - Average Loss: 0.2619
Epoch 16/30 - Average Loss: 0.2577
Epoch 17/30 - Average Loss: 0.2400
Epoch 18/30 - Average Loss: 0.2342
Epoch 19/30 - Average Loss: 0.2233
Epoch 20/30 - Average Loss: 0.2149
Epoch 21/30 - Average Loss: 0.2033
Epoch 22/30 - Average Loss: 0.1938
Epoch 23/30 - Average Loss: 0.1931
Epoch 24/30 - Average Loss: 0.1905
Epoch 25/30 - Average Loss: 0.1831
Epoch 26/30 - Average Loss: 0.1696
Epoch 27/30 - Average Loss: 0.1713
Epoch 28/30 - Average Loss: 0.1615
Epoch 29/30 - Average Loss: 0

In [9]:
torch.save(model.state_dict(), "face_model.pth")
print("Model saved successfully")

Model saved successfully


In [26]:
conn = sqlite3.connect("face_database.db")
cursor = conn.cursor()

print("Database cleared!")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS face_embeddings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            embedding BLOB NOT NULL
    )
""")

# Clearing the old incompatible embeddings
cursor.execute("DELETE FROM face_embeddings")
conn.commit()

print("Database ready!")

Database cleared!
Database ready!


In [27]:
def register_face(name, image_tensors, model, conn):
    model.eval()

    if not isinstance(image_tensors, list):
        image_tensors = [image_tensors]
    
    all_embeddings = []

    with torch.no_grad():
        for image in image_tensors:
            current_image = image.unsqueeze(0)
            embedding = model(current_image)
            all_embeddings.append(embedding)
    
    stacked = torch.stack(all_embeddings)
    average_embeddings = stacked.mean(dim = 0)

    average_embeddings = F_nn.normalize(average_embeddings, p=2, dim=1)

    embedding_bytes = pickle.dumps(average_embeddings)
    cursor = conn.cursor()
    cursor.execute("INSERT INTO face_embeddings (name, embedding) VALUES (?, ?)", (name, embedding_bytes))
    conn.commit()

    print(f"Registered: {name} (using {len(image_tensors)} image(s))")


In [28]:
def recognize_face(image_tensor, model, conn, threshold=0.8):
    model.eval()
    with torch.no_grad():
        batched_image = image_tensor.unsqueeze(0)
        new_embedding = model(batched_image)
    
    cursor = conn.cursor()
    cursor.execute("SELECT name, embedding FROM face_embeddings")
    rows = cursor.fetchall()

    if len(rows) == 0:
        return "No faces registered yet!"

    best_match = "Unknown"
    best_distance = float('inf')

    for name, embedding_bytes in rows:
        stored_embedding = pickle.loads(embedding_bytes)

        distance = torch.dist(new_embedding, stored_embedding).item()

        if distance < best_distance:
            best_match = name
            best_distance = distance

    if best_distance > threshold:
        return f"Unknown (closest was {best_match} at distance {best_distance:.4f})"
    
    return f"Match: {best_match} (distance: {best_distance:.4f})"

In [29]:
model = FaceEmbeddingNet()
model.load_state_dict(torch.load("face_model.pth"))
model.eval()

photo_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

In [35]:
img1 = Image.open(r"C:\Users\adty2\OneDrive\Pictures\PICTURES\WhatsApp Image 2025-04-18 at 18.24.01_9f2d194a.jpg")
img2 = Image.open(r"C:\Users\adty2\OneDrive\Pictures\PICTURES\WhatsApp Image 2025-04-18 at 18.24.01_caa9c50d.jpg")

tensor_img1 = photo_transform(img1)
tensor_img2 = photo_transform(img2)

register_face("Me", tensor_img1, model, conn)

result = recognize_face(tensor_img2, model, conn)
print(f"\nRecognition Result: {result}")


Registered: Me (using 1 image(s))

Recognition Result: Unknown (closest was Me at distance 0.9438)


In [31]:
img3 = Image.open(r"C:\Users\adty2\Downloads\pooja_image_for_db.jpeg")
img4 = Image.open(r"C:\Users\adty2\Downloads\harsh_image_for_db.jpeg")
img5 = Image.open(r"C:\Users\adty2\Downloads\Garima_image_for_db.jpeg")

tensor_img3 = photo_transform(img3)
tensor_img4 = photo_transform(img4)
tensor_img5 = photo_transform(img5)

register_face("Pooja", tensor_img3, model, conn)
register_face("Harsh", tensor_img4, model, conn)
register_face("Garima", tensor_img5, model, conn)


Registered: Pooja (using 1 image(s))
Registered: Harsh (using 1 image(s))
Registered: Garima (using 1 image(s))


In [32]:
img6 = Image.open(r"C:\Users\adty2\Downloads\pooja_image_for_test.jpeg")
img7 = Image.open(r"C:\Users\adty2\Downloads\harsh_image_for_test.jpeg")
img8 = Image.open(r"C:\Users\adty2\Downloads\Garima_image_for_test.jpeg")

tensor_img6 = photo_transform(img6)
tensor_img7 = photo_transform(img7)
tensor_img8 = photo_transform(img8)

result = recognize_face(tensor_img6, model, conn)
print(f"\nRecognition Result: {result}")

result = recognize_face(tensor_img7, model, conn)
print(f"\nRecognition Result: {result}")

result = recognize_face(tensor_img8, model, conn)
print(f"\nRecognition Result: {result}")


Recognition Result: Match: Pooja (distance: 0.5260)

Recognition Result: Match: Pooja (distance: 0.4342)

Recognition Result: Match: Garima (distance: 0.6682)
